This notebook checks the code for importing the disk dictionary and serves as a place to document notes and sources for the values included in the dictionary.

In [11]:
# add the host disk properties 
import pickle

with open(r'd:\CPD_MPIA\CPD_Emission_Models\disk_arr.pkl', 'rb') as f:
    disk_arr = pickle.load(f)

In [ ]:


with open(r'd:\CPD_MPIA\Median_SNR\all_disks.pkl', "rb") as f:
    all_disks = pickle.load(f)

all_disk_dicts = {}

for disk_name, disk_obj in all_disks.items():
    d = {}
    d['name'] = disk_name
    d['label'] = disk_name.replace('_', ' ')
    d['distance'] = getattr(disk_obj, 'distance_pc', None)
    d['mstar'] = getattr(disk_obj, 'stellar_mass', None)
    d['incl'] = getattr(disk_obj, 'inc', None)
    d['PA'] = getattr(disk_obj, 'PA', None)
    d['dx'], d['dy'] = getattr(disk_obj, 'center', (None, None))
    d['rout'] = getattr(disk_obj, 'disksize', {}).get('R95', None) if hasattr(disk_obj, 'disksize') else None   # in arcsec

    # Add info from disk_arr if available
    if disk_name in disk_arr:
        arr = disk_arr[disk_name]
        # Example: add Lstar and beam info if present
        if len(arr) > 3:
            d['Lstar'] = arr[3]
        if len(arr) > 4:
            d['beam_arcsec'] = arr[4]
        if len(arr) > 5:
            d['inc_disk_arr'] = arr[5]

    if hasattr(disk_obj, 'ringgap_info'):
        if hasattr(disk_obj, 'ringgap_info') and "flag" in disk_obj.ringgap_info:
            gaps = disk_obj.ringgap_info['flag'] == 0
            d['rgap'] = (disk_obj.ringgap_info['radius_arcsec'][gaps]/1000).tolist()   # mas
            d['wgap'] = (disk_obj.ringgap_info['width_arcsec'][gaps]/1000).tolist()   # mas
            d['dgap'] = (disk_obj.ringgap_info['gap_depth'][gaps]*100).tolist()  # percentage
    
    else:
        d['rgap'] = []
        d['wgap'] = []
        d['dgap'] = []
        d['rout'] = None


    #  Add Frank fitting parameters and same as DSHARP code
    d['hyp-alpha'] = 1.3
    d['hyp-wsmth'] = 0.1
    d['hyp-Ncoll'] = 300
    all_disk_dicts[disk_name] = d

with open("all_disk_dicts.pkl", "wb") as f:
    pickle.dump(all_disk_dicts, f)

In [13]:
with open("all_disk_dicts.pkl", "rb") as f:
    all_disk_dicts = pickle.load(f)

# Now you can access the dictionary:
#print(all_disk_dicts["AA_Tau"])

#### check with the Andrews Code

```
disk = {}

disk['SR4'] =      {'name': 'SR4',   
                    'label': 'SR 4', 
                    'distance': 134.8,
                    'mstar': 0.68,
                    'lstar': 1.17,
                    'incl': 22.0, 
                    'PA': 18.0,
                    'dx': -0.060,  # RA offset in arcsec
                    'dy': -0.509,  # Dec offset in arcsec
                    'rgap': [0.079],  # mas
                    'wgap': [0.010],  # mas
                    'dgap': [30],  # 30 percent flux drop
                    'rout': 0.25, 
                    'maxTb': 50,
                    'hyp-alpha': 1.3,   # frank fitting  - smoothing strength
                    'hyp-wsmth': 0.1, # frank fitting  - smoothing scale
                    'hyp-Ncoll': 300,  # frank fitting  - number of collocation points
                    'cmask': 'circle[[16h25m56.16s, -24.20.48.71], 0.7arcsec]',  # for casa tclean to define imaging region (not full fov)
                    'cscales': [0, 5, 30, 75, 150],    # in very simple words, these are the different "size" of structures that tclean will try to decompose the image into, structures of size ~scale will be modelled with that scale, in units of pixels
		    'gscales': [0, 5],   # for gap specific cleaning
                    'cthresh': '0.05mJy',   
                    'gthresh': '0.034mJy',
                    'crobust': -0.5,
                    'ctaper': ['0.035arcsec', '0.01arcsec', '0deg'],
                    'cgain': 0.3,
                    'ccycleniter': 300,
                    'RMS': 17.8,
                    'peakr': [84.], 
                    'peakaz': [104.]
}
```


    #### Imaging parameters
    1. 'maxTb' =  
    2. 'RMS' = 

    #### Frank fitting
    3. 'hyp-alpha' = 1.3
    4. 'hyp-wsmth' = 0.1
    5. 'hyp-Ncoll' = 300

    #### CASA Imaging Settings:

    6. 'cmask'  =    'circle[[16h25m56.16s, -24.20.48.71], 0.7arcsec]'
    7. 'cscales'
    8. 'gscales'
    9. 'cthresh'
    10. 'gthresh'
    11. 'crobust'
    12. 'ctaper'
    13. 'cgain'
    14. 'ccycleniter'
    15. 'RMS'

    #### Peak Detection
    16. 'peakr'
    17. 'peakaz'   


    #### minimmum needed for CPD search:
    disk['Elias20'] = {
    'name': 'Elias20',
    'incl': 54.0, 
    'PA': 153.2, 
    'dx': -0.052, 
    'dy': -0.490, 
    'rgap': [0.181],
    'wgap': [0.011],
    'rout': 0.48, 
    'hyp-alpha': 1.3,
    'hyp-wsmth': 0.1,
    'hyp-Ncoll': 300,
    'cmask': 'ellipse[[16h26m18.87s, -24.28.20.18], [0.8arcsec, 0.5arcsec], 154deg]',
    'cscales': [0, 10, 25, 50, 100],
    'cthresh': '0.06mJy',
    'crobust': 0.0,
    'ctaper': [],
    'cgain': 0.3,
    'ccycleniter': 300,
}

In [14]:
import pprint
pprint.pprint(all_disk_dicts)

{'AA_Tau': {'Lstar': 1.1,
            'PA': 93.77079777,
            'beam_arcsec': 0.12,
            'dgap': [0.01, 0.44, 0.34, 0.94],
            'distance': 135,
            'dx': -0.00545897,
            'dy': 0.00482739,
            'hyp-Ncoll': 300,
            'hyp-alpha': 1.3,
            'hyp-wsmth': 0.1,
            'inc_disk_arr': 58.54,
            'incl': 58.53531224,
            'label': 'AA Tau',
            'mstar': None,
            'name': 'AA_Tau',
            'rgap': [11.0, 64.3, 79.8, 105.3],
            'rout': np.float64(1.177),
            'wgap': [28.1, 8.2, 10.2, 4.9]},
 'CQ_Tau': {'Lstar': 10,
            'PA': 53.87180444,
            'beam_arcsec': 0.135,
            'dgap': [],
            'distance': 149,
            'dx': -0.00871044,
            'dy': 0.0009941,
            'hyp-Ncoll': 300,
            'hyp-alpha': 1.3,
            'hyp-wsmth': 0.1,
            'inc_disk_arr': 35.24,
            'incl': 35.2426038,
            'label': 'CQ Tau',
      